# DATA CLEANING

1. Importo le librerie necessarie e carico i file .csv

In [20]:
import pandas as pd
import seaborn as ss
import numpy as np
import matplotlib 
import matplotlib.pyplot as pp

In [21]:
calendar = pd.read_csv("calendar.csv")
customers = pd.read_csv("customers.csv")
products = pd.read_csv("products.csv")
sales = pd.read_csv("sales.csv")
stores = pd.read_csv("stores.csv")

2. Proseguo con un merge per avere un unico Dataframe per l'analisi

In [22]:
df = sales.merge(products, on='product_id') \
          .merge(stores, on='store_id') \
          .merge(customers, on='customer_id')

3. Controllo della tabella e dei data types

In [23]:
df.head()


,order_id,order_date,product_id,store_id,customer_id,quantity,unit_price,discount,revenue,cost,...,cocoa_percent,weight_g,store_name,city,country,store_type,age,gender,loyalty_member,join_date
0,0RD00000001,2023-01-07,P0080,S093,C040749,5,14.43,0.15,61.33,42.77,...,70,200,Chocolate Store 93,Sydney,UK,Airport,44,Male,1,2021-11-17
1,0RD00000002,2023-10-22,P0173,S065,C020161,3,12.01,0.00,36.03,19.06,...,60,50,Chocolate Store 65,New York,Australia,Retail,63,Female,1,2023-07-03
2,0RD00000003,2023-05-07,P0115,S078,C048069,2,10.02,0.00,20.04,10.29,...,90,50,Chocolate Store 78,London,UK,Airport,35,Male,1,2023-10-09
3,0RD00000004,2024-06-23,P0186,S088,C047901,2,14.66,0.10,26.39,16.35,...,60,50,Chocolate Store 88,Toronto,USA,Retail,37,Female,1,2023-05-30
4,0RD00000005,2024-09-24,P0197,S054,C033950,1,12.34,0.00,12.34,7.94,...,90,120,Chocolate Store 54,London,Canada,Online,57,Female,0,2021-08-20


In [24]:
df.dtypes

order_id           object
order_date         object
product_id         object
store_id           object
customer_id        object
quantity            int64
unit_price        float64
discount          float64
revenue           float64
cost              float64
profit            float64
product_name       object
brand              object
category           object
cocoa_percent       int64
weight_g            int64
store_name         object
city               object
country            object
store_type         object
age                 int64
gender             object
loyalty_member      int64
join_date          object
dtype: object

    Nel Dataset ci sono delle incongruenze

4.  Switch dtypes

    La colonna Loyalty_member essendo "Int64" potrebbe suggerire che i valori possano essere sommati, 
    
    ma i dati nella tabella sono 0 e 1, cioè Sì e NO, quindi cambio il dtype in bool per prevenire errori ed avere una maggiore chiarezza.

    La colonna order_date la rendo di tipo date

    Cocoa_percent per consistenza e coerenza lo rendo float (es. 0.7 anzichè 70)


In [25]:
df['loyalty_member'] = df['loyalty_member'].astype(bool)
df['order_date'] = pd.to_datetime(df['order_date'])
df['cocoa_percent'] = df['cocoa_percent'] / 100

5. Correzione city-country

    Nella tabella stores, le città non corrispondono allo stato di appartenenza (es. Berlin - France / Sydney - UK), quindi aggiorno il dizionario.

In [26]:
error_check = df[['city', 'country']].drop_duplicates().sort_values('city')
print(error_check)

          city    country
127     Berlin    Germany
36      Berlin         UK
14      Berlin     France
77      Berlin        USA
72      Berlin  Australia
31      London    Germany
17      London     France
11      London  Australia
134     London        USA
4       London     Canada
2       London         UK
7    Melbourne        USA
78   Melbourne     France
38   Melbourne         UK
22   Melbourne     Canada
8    Melbourne  Australia
9    Melbourne    Germany
33    New York         UK
71    New York    Germany
43    New York        USA
218   New York     France
28    New York     Canada
1     New York  Australia
116      Paris  Australia
81       Paris        USA
5        Paris     Canada
6        Paris     France
52       Paris    Germany
66       Paris         UK
0       Sydney         UK
57      Sydney    Germany
50      Sydney     Canada
13      Sydney        USA
40      Sydney     France
18      Sydney  Australia
29     Toronto  Australia
10     Toronto     Canada
3      Toron

In [27]:
cities = df['city'].unique()    #recupero le città per non cercarle "manualmente"
pippo = {city: "" for city in cities}
print(pippo)

{'Sydney': '', 'New York': '', 'London': '', 'Toronto': '', 'Paris': '', 'Melbourne': '', 'Berlin': ''}


In [28]:
city_to_country = {
    'Sydney': 'Australia',
    'New York': 'USA',
    'London': 'UK',
    'Toronto': 'Canada',
    'Melbourne': 'Australia',
    'Paris': 'France',
    'Berlin': 'Germany'
    }

In [29]:
df['country'] = df['city'].map(city_to_country).fillna(df['country'])


In [30]:
print(df[['city', 'country']].drop_duplicates())

         city    country
0      Sydney  Australia
1    New York        USA
2      London         UK
3     Toronto     Canada
5       Paris     France
7   Melbourne  Australia
14     Berlin    Germany


In [31]:
df.head()

,order_id,order_date,product_id,store_id,customer_id,quantity,unit_price,discount,revenue,cost,...,cocoa_percent,weight_g,store_name,city,country,store_type,age,gender,loyalty_member,join_date
0,0RD00000001,2023-01-07,P0080,S093,C040749,5,14.43,0.15,61.33,42.77,...,0.7,200,Chocolate Store 93,Sydney,Australia,Airport,44,Male,True,2021-11-17
1,0RD00000002,2023-10-22,P0173,S065,C020161,3,12.01,0.00,36.03,19.06,...,0.6,50,Chocolate Store 65,New York,USA,Retail,63,Female,True,2023-07-03
2,0RD00000003,2023-05-07,P0115,S078,C048069,2,10.02,0.00,20.04,10.29,...,0.9,50,Chocolate Store 78,London,UK,Airport,35,Male,True,2023-10-09
3,0RD00000004,2024-06-23,P0186,S088,C047901,2,14.66,0.10,26.39,16.35,...,0.6,50,Chocolate Store 88,Toronto,Canada,Retail,37,Female,True,2023-05-30
4,0RD00000005,2024-09-24,P0197,S054,C033950,1,12.34,0.00,12.34,7.94,...,0.9,120,Chocolate Store 54,London,UK,Online,57,Female,False,2021-08-20


6. Creazione nuove colonne ed estrazione dati temporali

In [37]:
df['unit_cost'] = (df['cost'] / df['quantity']).round(2)
df['profit'] = (df['revenue'] - (df['quantity'] * df['unit_cost'])).round(2)
df['margin_percentage'] = ((df['profit'] / df['revenue']) * 100).round(2)
df['revenue'] = df['revenue'].round(2)

In [41]:
df[['profit', 'margin_percentage', 'revenue', 'quantity', 'unit_price', 'unit_cost', 'cost']].head()

,profit,margin_percentage,revenue,quantity,unit_price,unit_cost,cost
0,18.58,30.30,61.33,5,14.43,8.55,42.77
1,16.98,47.13,36.03,3,12.01,6.35,19.06
2,9.76,48.70,20.04,2,10.02,5.14,10.29
3,10.03,38.01,26.39,2,14.66,8.18,16.35
4,4.40,35.66,12.34,1,12.34,7.94,7.94


In [35]:
df['month'] = df['order_date'].dt.month
df['year'] = df['order_date'].dt.year

7. Esportazione in file .csv

In [43]:
df.to_csv("df_cleaned_chocolate.csv", index=False)